In [ ]:
import numpy as np
import pandas as pd

### 1. Dataset Loading

In [ ]:
charts_raw = pd.read_csv("../data/eu_charts.csv")
charts = charts_raw.copy()
charts.info()

### 2. String to date conversion

In [ ]:
charts['date'] = pd.to_datetime(charts['date'], errors='coerce')

### 3. Track ID extraction
- We want to merge 2 or more datasets, so we need a common key. 
- The "url" column contains Spotify track URLs, which include the track ID. 
-  We can extract the track ID from the URL to create a new "track_id" column that will serve as a common key for merging with the tracks dataset.

In [ ]:
charts['url'].isna().sum()

In [ ]:
# Example url: https://open.spotify.com/track/2ZywW3VyVx6rrrrX75n3JB?si=xxxx
charts["track_id"] = (
    charts["url"]
      .str.split("/track/", n=1).str[-1]   # keep part after "/track/"
      .str.split("?", n=1).str[0]          # remove query params
      .str.strip()
)

### 4. Tracks Dataset Loading

In [ ]:
tracks_raw = pd.read_csv("../data/tracks_clean.csv")
tracks = tracks_raw.copy()
tracks.info()

In [ ]:
tracks = tracks.rename(columns={"id": "track_id"})
charts = charts.rename(columns={"region": "country"})

### 5. Datasets Join

In [ ]:
merged_df = charts.merge(tracks, on="track_id", how="left")
merged_df

In [ ]:
merged_df.info()

In [ ]:
# Check missing values in "tempo" column by country
merged_df.groupby('country')['tempo'].apply(lambda x: x.isna().sum() / len(x))

⚠️ There are too many missing audio features in the merged dataset. 
So instead of a left join (more accurate analysis) we'll switch to a inner join instead.
We could consider augmenting the dataset using: https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset

In [ ]:
inner_merged_df = charts.merge(tracks, on="track_id", how="inner")
inner_merged_df

In [ ]:
inner_merged_df.groupby('country')['valence'].apply(lambda x: x.isna().sum() / len(x))

In [ ]:
inner_merged_df.info()

### 6. Final cleaning and export

In [ ]:
# remove useless columns
inner_merged_df.drop(columns=['artist', 'url', 'trend', 'name', 'release_date', 'year'], inplace=True)

In [ ]:
""" Df split based on 'chart' since viral50 is based on virality (how many times a song is shared or added to favorites) not on streams quantity
top200_eu = inner_merged_df[inner_merged_df['chart'] == 'top200']
viral50_eu = inner_merged_df[inner_merged_df['chart'] == 'viral50']"""

In [ ]:
inner_merged_df.to_csv("../data/eu_charts_with_tracks_data.csv", index=False)